<a href="https://colab.research.google.com/github/cojocarucosmin/AICourseDev/blob/main/State_of_the_art_Metadata_Document_Extractor_with_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Parsing and extracting metadata using OCR, HF Transformer and ChatGPT**

In [1]:
#@title # Cell 1: Installations
!pip install -q gradio openai pymupdf pymupdf4llm transformers torch torchvision torchaudio pandas numpy Pillow tiktoken openpyxl pytesseract
!sudo apt-get update && sudo apt-get install -y tesseract-ocr  && sudo apt-get clean
print("Installations complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 36.6 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,065 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRe

In [ ]:
#@title # Cell 2: Imports, Configuration, and Prompts
import os, io, glob
from pathlib import Path
import json
import time
import math
import hashlib
import shutil
import traceback
from datetime import datetime

import gradio as gr
import fitz  # PyMuPDF
import pymupdf4llm  # For Markdown conversion
import pytesseract
from PIL import Image
import pandas as pd
import openai
import tiktoken
import numpy as np
import torch
from transformers import AutoImageProcessor, AutoModelForObjectDetection

# --- Tesseract CONFIGURATION ---
# Uncomment and set the correct tesseract path if needed:
# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'  # For Linux
# pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'  # For Windows

# --- CONSTANTS ---
DEFAULT_MODEL = "gpt-5-mini"
CHUNK_THRESHOLD = 80000     # Token threshold to trigger chunking (adjust based on model & testing)
MAX_CHUNK_TOKENS = 70000    # Target max tokens per chunk in Map phase (leave buffer)
REDUCE_TOKEN_LIMIT = 100000  # Max tokens for the Reduce phase input (adjust based on model)

# Use pathlib to define an absolute cache directory for robustness.
BASE_DIR = Path(os.getcwd())
CACHE_DIR = BASE_DIR / "metadata_cache"
OUTPUT_DIR = BASE_DIR / "metadata_output"

# Ensure directories exist
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- PRICING INFORMATION (Apr 2025) ---
PRICING_DATA = {
    "last_updated": "Aug 2025",
    "prices": {
        "gpt-5":       {"input": 1.25,   "output": 10.0},
        "gpt-5-mini":  {"input": 0.25,   "output": 2.0},
        "gpt-5-nano":  {"input": 0.05,   "output": 0.40},
        "gpt-4o":      {"input": 2.50,   "output": 10.0},
        "o4-mini":     {"input": 1.10,   "output": 4.40}
    }
}
model_choices = list(PRICING_DATA["prices"].keys())

# --- BASE SYSTEM PROMPT ---
BASE_SYSTEM_PROMPT = """You are an expert AI assistant specialized in extracting structured metadata from documents. The input provided is a JSON object containing two keys:
1.  'document_markdown': Holds the main text and tables from the document converted to Markdown (or plain text if chunked).
2.  'image_content': A list of objects, where each object represents content extracted (via OCR or table recognition) from an image within the document, including its page number and type (e.g., 'ocr_text', 'markdown_table').

Analyze **both** the 'document_markdown' content and the 'image_content' data comprehensively. Your goal is to generate a **single, valid JSON object** containing the structured metadata based on the user's requested schema (provided below).

**Instructions:**
- Adhere strictly to the JSON schema requested by the user.
- If information for a specific field requested by the user is not found in either the markdown or the image content, use `null` for optional fields or an empty string (`""`) / empty list (`[]`) as appropriate for the required field type defined in the user's schema.
- Ensure the final output is ONLY the JSON object, with no introductory text, explanations, or markdown formatting (like ```json).

**User's Requested JSON Schema:**
"""

# --- USER-EDITABLE SYSTEM PROMPT ---
SYSTEM_PROMPT = """
Extract the following fields:
1.  `title`: The main title of the document. (Type: string)
2.  `journal_conference`: The name of the journal or conference proceedings. (Type: string/null)
3.  `doi`: The Digital Object Identifier. (Type: string/null)
4.  `authors`: A list of author names. (Type: list of strings)
5.  `publication_year`: The year the document was published. (Type: integer/null)
6.  `keywords`: A list of keywords provided in the document. (Type: list of strings)
7.  `abstract`: The abstract section of the document. (Type: string/null)
8.  `summary`: A brief summary generated by you based on the content (especially abstract/conclusions). (Type: string)
9.  `conclusions`: The main conclusions stated in the document. (Type: string/null)
10. `advantages_disadvantages`: Mentioned advantages and disadvantages of the methods/approach. (Type: object with keys "advantages" (list of strings) and "disadvantages" (list of strings))
11. `explainable_ai_mentions`: Specific text snippets discussing Explainable AI (XAI) or interpretability. (Type: list of strings)
12. `future_work_next_steps`: Stated future work or next steps. (Type: list of strings)
13. `region_focus`: Any specific geographical region focus mentioned. (Type: string/null)
14. `dataset_details`: Information about datasets used (e.g., name, size, source). (Type: string/null)
15. `data_types_used`: Types of data analyzed (e.g., text, image, tabular, time-series). (Type: list of strings)
16. `company_types_mentioned`: Types of companies relevant to the study (if any). (Type: list of strings)
17. `time_period_analyzed`: Specific time periods covered by the data or study. (Type: string/null)
18. `models_approach_used`: Key models, algorithms, or approaches employed. (Type: list of strings)
19. `innovative_aspects`: Highlight any particularly innovative algorithms or techniques mentioned. (Type: list of strings)
20. `general_applicability`: Assessment of the general applicability of the findings/methods. (Type: string/null)
21. `performance_metrics`: Key performance results reported for models/methods. (Type: object or list of objects, e.g., {"model": "X", "metric": "Accuracy", "value": "95%"})

Remember: Output **only** the final JSON object. No extra text, explanations, or markdown formatting.
"""

print("Imports, Configuration, and Prompts loaded.")


Imports, Configuration, and Prompts loaded.


In [ ]:
#@title # Cell 3: Hugging Face Model Loading
print("Loading Hugging Face model and processor...")

# Determine device: use GPU if available, otherwise fall back to CPU.
hf_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if hf_device.type == "cuda":
    print("GPU is available. Using GPU.")
else:
    print("GPU not available. Using CPU (this will be slower).")

model_checkpoint = "microsoft/table-transformer-structure-recognition"

try:
    hf_image_processor = AutoImageProcessor.from_pretrained(model_checkpoint)
    hf_model_obj_detect = AutoModelForObjectDetection.from_pretrained(model_checkpoint)
    hf_model_obj_detect.to(hf_device)
    hf_model_loaded = True
    print("Hugging Face Model and processor loaded successfully.")
except Exception as e:
    hf_model_loaded = False
    hf_model_obj_detect = None
    hf_image_processor = None
    print(f"❌ WARNING: Could not load Hugging Face model: {e}")
    print("Table extraction from images will fall back to basic OCR only.")

In [ ]:
#@title # Cell 4: Helper Functions (HF Table Extraction, Token Counting)

# --- HF Helper Function ---
def extract_table_from_image_hf(image: Image.Image, model, processor, device,
                                row_conf=0.9, col_conf=0.9, ocr_psm=7):
    """
    Extracts a table from a PIL Image using the Hugging Face Table Transformer.
    Returns a pandas DataFrame if extraction is successful; otherwise returns None.

    Parameters:
      - image: PIL.Image.Image object.
      - model: The loaded HF model.
      - processor: The corresponding image processor.
      - device: Torch device (CPU/GPU) used.
      - row_conf: Confidence threshold for row detections.
      - col_conf: Confidence threshold for column detections.
      - ocr_psm: Page segmentation mode for Tesseract OCR (optional, default psm=7).
    """
    # Check if HF model is loaded; if not, return None immediately.
    if not hf_model_loaded or model is None or processor is None:
        return None

    start_time = time.time()
    try:
        # Preprocess the image
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        model.eval()
        with torch.no_grad():
            outputs = model(**inputs)

        width, height = image.size
        detection_threshold = min(row_conf, col_conf) - 0.1

        # Post-process detection results to obtain bounding boxes, labels, and scores.
        results = processor.post_process_object_detection(
            outputs,
            threshold=detection_threshold,
            target_sizes=[(height, width)]
        )[0]

        boxes = results['boxes'].cpu().numpy()
        raw_label_ids = results['labels'].cpu().numpy()
        scores = results['scores'].cpu().numpy()
        id2label = model.config.id2label

        # Identify the label IDs corresponding to "table row" and "table column"
        row_label_id = next((id for id, label in id2label.items() if label == 'table row'), None)
        col_label_id = next((id for id, label in id2label.items() if label == 'table column'), None)
        if row_label_id is None or col_label_id is None:
            return None

        # Filter boxes based on confidence thresholds and the label type.
        row_boxes = [box for box, label_id, score in zip(boxes, raw_label_ids, scores)
                     if label_id == row_label_id and score > row_conf]
        col_boxes = [box for box, label_id, score in zip(boxes, raw_label_ids, scores)
                     if label_id == col_label_id and score > col_conf]
        if not row_boxes or not col_boxes:
            return None

        # Sort the boxes so that they are in proper order for rows and columns.
        row_boxes.sort(key=lambda box: box[1])
        col_boxes.sort(key=lambda box: box[0])

        table_data = []
        for r_idx, r_box in enumerate(row_boxes):
            row_text = []
            for c_idx, c_box in enumerate(col_boxes):
                # Determine the overlapping region between the current row and column boxes.
                x1, y1 = max(r_box[0], c_box[0]), max(r_box[1], c_box[1])
                x2, y2 = min(r_box[2], c_box[2]), min(r_box[3], c_box[3])
                cell_text = ""
                if x1 < x2 and y1 < y2:
                    intersection_box = [int(x1), int(y1), int(x2), int(y2)]
                    try:
                        # Crop the cell from the image and use Tesseract for OCR.
                        cell_img = image.crop(intersection_box)
                        ocr_config = f'--psm 7 -c tessedit_char_whitelist=0123456789.,abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ%()- /$£€:'
                        cell_text = pytesseract.image_to_string(cell_img, config=ocr_config).strip()
                        if not cell_text:
                            cell_text = ""
                    except Exception as ocr_err:
                        cell_text = "[OCR Err]"
                row_text.append(cell_text)
            table_data.append(row_text)

        # Convert the collected table data into a DataFrame and remove any empty rows/columns.
        df = pd.DataFrame(table_data)
        df.dropna(axis=0, how='all', inplace=True)
        df.dropna(axis=1, how='all', inplace=True)

        # Return the DataFrame if it contains data, otherwise return None.
        return df if not df.empty else None
    except Exception as hf_err:
        print(f"Error during HF table extraction: {hf_err}")
        return None

# --- TOKEN COUNTING FUNCTION ---
def count_tokens(text, model):
    """
    Counts the number of tokens in the provided text using tiktoken.
    Falls back to word count if tiktoken fails to find encoding for the model.

    Parameters:
      - text: The input text (or any object convertible to string).
      - model: The model name to use for tokenization.

    Returns:
      - Integer count of tokens.
    """
    model_name_for_tiktoken = model
    # Map common model names to tiktoken compatible names
    if model not in ["gpt-4o", "gpt-4o-mini", "gpt-3.5-turbo"]:
        if "gpt-4" in model:
            model_name_for_tiktoken = "gpt-4"
        elif "gpt-3.5" in model:
            model_name_for_tiktoken = "gpt-3.5-turbo"
        else:
            model_name_for_tiktoken = "gpt-4"  # Default fallback

    try:
        encoding = tiktoken.encoding_for_model(model_name_for_tiktoken)
        return len(encoding.encode(str(text)))
    except KeyError:
        print(f"Warning: Tiktoken model '{model_name_for_tiktoken}' not found. Falling back to word count.")
        return len(str(text).split())
    except Exception as e:
        print(f"Warning: Tiktoken error ({e}). Falling back to word count.")
        return len(str(text).split())

print("Helper functions defined.")


Helper functions defined.


In [ ]:
#@title # Cell 5: Content Extraction Functions

# --- FILE HASHING FUNCTION ---
def get_file_hash(filepath):
    """Calculates the SHA256 hash of a file."""
    hasher = hashlib.sha256()
    try:
        with open(filepath, 'rb') as f:
            # Read file in 4096-byte chunks until EOF
            for chunk in iter(lambda: f.read(4096), b""):
                hasher.update(chunk)
    except Exception as e:
        print(f"Error reading file for hash computation: {e}")
        return None
    return hasher.hexdigest()

print("Hashing function defined.")

# --- CONTENT EXTRACTION (Whole Document) ---
def extract_structured_content(pdf_path, log_list):
    """
    Extracts document content as markdown and processes images via HF Table Extraction or Tesseract OCR.
    Returns a dictionary with:
      - "document_markdown": extracted text converted to Markdown.
      - "image_content": list of extracted image data.
    """
    try:
        filename = os.path.basename(pdf_path)
        log_list.append(f"Extracting markdown from {filename}...")
        # Use pymupdf4llm to extract markdown content from the PDF
        md_text = pymupdf4llm.to_markdown(pdf_path, write_images=False).strip()

        log_list.append(f"Processing images in {filename}...")
        image_content_results = []
        images_processed_count = 0

        # Open PDF with fitz using a with-statement for proper closure
        with fitz.open(pdf_path) as doc:
            num_pages = len(doc)
            for page_num, page in enumerate(doc, start=1):
                images = page.get_images(full=True)
                if not images:
                    continue
                for img_index, img in enumerate(images, start=1):
                    xref = img[0]
                    content_type = "error"
                    content = f"Failed to process image xref {xref}"

                    try:
                        base_image = doc.extract_image(xref)
                        if not base_image or not base_image.get("image"):
                            log_list.append(f"Skipping image {img_index} on page {page_num} (xref {xref}): No base image data.")
                            continue

                        image_bytes = base_image["image"]
                        pil_image = Image.open(io.BytesIO(image_bytes))
                        if pil_image.mode != 'RGB':
                            pil_image = pil_image.convert('RGB')
                        images_processed_count += 1

                        # Attempt table extraction via HF. If unsuccessful, fall back to Tesseract OCR.
                        table_df = extract_table_from_image_hf(pil_image, hf_model_obj_detect, hf_image_processor, hf_device)
                        if table_df is not None:
                            content_type = "markdown_table"
                            content = table_df.to_markdown(index=False)
                        else:
                            content = pytesseract.image_to_string(pil_image).strip()
                            content_type = "ocr_text"
                            if not content:
                                content = "[No text found by OCR]"

                        image_content_results.append({
                            "page_num": page_num,
                            "image_index": img_index,
                            "content_type": content_type,
                            "content": content
                        })
                    except Image.UnidentifiedImageError:
                        log_list.append(f"Warning: Skipping image {img_index} on page {page_num} (xref {xref}): Unidentified image format.")
                        image_content_results.append({
                            "page_num": page_num,
                            "image_index": img_index,
                            "content_type": "error",
                            "content": "Unidentified image format"
                        })
                    except Exception as e:
                        log_list.append(f"Warning: Could not process image {img_index} on page {page_num} (xref {xref}): {e}")
                        image_content_results.append({
                            "page_num": page_num,
                            "image_index": img_index,
                            "content_type": "error",
                            "content": f"Error processing image: {str(e)[:200]}"
                        })
            log_list.append(f"Finished processing {images_processed_count} images across {num_pages} pages.")
        # <-- Updated key below: use "document_markdown" to match our prompt and cache check
        return {"document_markdown": md_text, "image_content": image_content_results}
    except Exception as e:
        log_list.append(f"❌ Error during PDF content extraction ({os.path.basename(pdf_path)}): {e}")
        log_list.append(traceback.format_exc())
        return None

# --- CONTENT EXTRACTION (Per Page) ---
def extract_content_per_page(pdf_path, log_list):
    """
    Extracts content page by page from a PDF, capturing both text and images.

    Returns:
      list: Each element is a dict with keys: 'page_num', 'text', and 'image_content'.
    """
    pages_data = []
    images_processed_count = 0
    try:
        filename = os.path.basename(pdf_path)
        log_list.append(f"Extracting content per page from {filename}...")

        with fitz.open(pdf_path) as doc:
            num_pages = len(doc)
            for page_num, page in enumerate(doc, start=1):
                page_text = page.get_text("text").strip()
                page_image_content = []
                images = page.get_images(full=True)
                if images:
                    for img_index, img in enumerate(images, start=1):
                        xref = img[0]
                        content_type = "error"
                        content = f"Failed to process image xref {xref}"
                        try:
                            base_image = doc.extract_image(xref)
                            if not base_image or not base_image.get("image"):
                                log_list.append(f"Skipping image {img_index} on page {page_num} (xref {xref}): No base image data.")
                                continue

                            image_bytes = base_image["image"]
                            pil_image = Image.open(io.BytesIO(image_bytes))
                            if pil_image.mode != 'RGB':
                                pil_image = pil_image.convert('RGB')
                            images_processed_count += 1

                            table_df = extract_table_from_image_hf(pil_image, hf_model_obj_detect, hf_image_processor, hf_device)
                            if table_df is not None:
                                content_type = "markdown_table"
                                content = table_df.to_markdown(index=False)
                            else:
                                content = pytesseract.image_to_string(pil_image).strip()
                                content_type = "ocr_text"
                                if not content:
                                    content = "[No text found by OCR]"

                            page_image_content.append({
                                "page_num": page_num,
                                "image_index": img_index,
                                "content_type": content_type,
                                "content": content
                            })
                        except Image.UnidentifiedImageError:
                            log_list.append(f"Warning: Skipping image {img_index} on page {page_num} (xref {xref}): Unidentified image format.")
                            page_image_content.append({
                                "page_num": page_num,
                                "image_index": img_index,
                                "content_type": "error",
                                "content": "Unidentified image format"
                            })
                        except Exception as e:
                            log_list.append(f"Warning: Could not process image {img_index} on page {page_num} (xref {xref}): {e}")
                            page_image_content.append({
                                "page_num": page_num,
                                "image_index": img_index,
                                "content_type": "error",
                                "content": f"Error processing image: {str(e)[:200]}"
                            })
                pages_data.append({
                    "page_num": page_num,
                    "text": page_text,
                    "image_content": page_image_content
                })
            log_list.append(f"Finished processing {images_processed_count} images across {num_pages} pages.")
        return pages_data
    except Exception as e:
        log_list.append(f"❌ Error during per-page content extraction ({os.path.basename(pdf_path)}): {e}")
        log_list.append(traceback.format_exc())
        return None

print("Content extraction functions defined.")

Hashing function defined.
Content extraction functions defined.


In [ ]:
#@title # Cell 6: Chunking Function

def create_document_chunks(pages_data, system_prompt, model, max_chunk_tokens, log_list, overlap_count=5):
    """
    Groups pages into chunks based on estimated token count with an overlap.

    Each chunk is a dict with two keys:
      - "pages": a list of page numbers included in the chunk.
      - "content_dict": a dict with keys "document_markdown" and "image_content" that aggregates
                        the text and image data from those pages.

    Parameters:
      - pages_data: list of dicts, one per page (each having keys 'page_num', 'text', and 'image_content').
      - system_prompt: The prompt used to count baseline tokens (e.g., the combined prompt for estimation).
      - model: The OpenAI model identifier to use for token counting.
      - max_chunk_tokens: The target maximum number of tokens per chunk.
      - log_list: A list where log messages are appended.
      - overlap_count: Number of pages to overlap between consecutive chunks (default 5).

    Returns:
      - A list of chunk dicts.
    """
    from copy import deepcopy

    def aggregate_pages(pages):
        """Aggregate text and image content from a list of page dicts and compute token count."""
        agg_md = ""
        agg_imgs = []
        total_tokens = 0
        for p in pages:
            text = p.get("text", "")
            imgs = p.get("image_content", [])
            agg_md += text + "\n\n"
            agg_imgs.extend(imgs)
            total_tokens += count_tokens(text, model)
            total_tokens += count_tokens(json.dumps(imgs), model)
        return {"document_markdown": agg_md, "image_content": agg_imgs}, total_tokens

    chunks = []
    if not pages_data:
        log_list.append("❌ Chunking Error: No page data provided.")
        return chunks

    # Use full page dictionaries for accumulation.
    current_chunk_pages = []  # Each element is a full page dict
    # Start token count with tokens from the provided system prompt.
    system_prompt_tokens = count_tokens(system_prompt, model)
    current_token_estimate = system_prompt_tokens
    log_list.append(f"Chunking: System prompt tokens = {system_prompt_tokens}")

    for page in pages_data:
        current_chunk_pages.append(page)
        # Aggregate current chunk and update token count
        aggregated, pages_tokens = aggregate_pages(current_chunk_pages)
        current_token_estimate = system_prompt_tokens + pages_tokens

        # If adding this page pushes tokens over the threshold and we have more than one page:
        if current_token_estimate > max_chunk_tokens and len(current_chunk_pages) > 1:
            # Remove the last page (the one that caused the overshoot)
            overshot_page = current_chunk_pages.pop()
            aggregated, pages_tokens = aggregate_pages(current_chunk_pages)
            current_token_estimate = system_prompt_tokens + pages_tokens
            log_list.append(f"Finalizing chunk {len(chunks)+1} with pages {[p.get('page_num') for p in current_chunk_pages]} (~{current_token_estimate} tokens, limit {max_chunk_tokens}).")
            chunks.append({
                "pages": [p.get("page_num") for p in current_chunk_pages],
                "content_dict": deepcopy(aggregated)
            })
            # Prepare for next chunk: retain the last 'overlap_count' pages
            if len(current_chunk_pages) >= overlap_count:
                current_chunk_pages = deepcopy(current_chunk_pages[-overlap_count:])
            else:
                current_chunk_pages = deepcopy(current_chunk_pages)
            aggregated, pages_tokens = aggregate_pages(current_chunk_pages)
            current_token_estimate = system_prompt_tokens + pages_tokens

        # Edge-case: if a single page (with system prompt) exceeds max_chunk_tokens, finalize it as standalone.
        elif len(current_chunk_pages) == 1 and (system_prompt_tokens + current_token_estimate - system_prompt_tokens) > max_chunk_tokens:
            log_list.append(f"⚠️ Warning: Page {page.get('page_num', 'Unknown')} alone exceeds token limit (~{system_prompt_tokens + current_token_estimate} tokens). Processing it as a standalone chunk.")
            chunks.append({
                "pages": [page.get("page_num")],
                "content_dict": deepcopy({"document_markdown": page.get("text", "") + "\n\n", "image_content": page.get("image_content", [])})
            })
            current_chunk_pages = []
            current_token_estimate = system_prompt_tokens

    # Finalize any remaining pages as the last chunk.
    if current_chunk_pages:
        aggregated, pages_tokens = aggregate_pages(current_chunk_pages)
        current_token_estimate = system_prompt_tokens + pages_tokens
        log_list.append(f"Finalizing last chunk {len(chunks)+1} with pages {[p.get('page_num') for p in current_chunk_pages]} (~{current_token_estimate} tokens).")
        chunks.append({
            "pages": [p.get("page_num") for p in current_chunk_pages],
            "content_dict": deepcopy(aggregated)
        })

    log_list.append(f"Document split into {len(chunks)} chunk(s).")
    return chunks


# --- Save Outputs Helper Function ---
def save_outputs_to_disk(results_obj, timestamp):
    """
    Saves the extracted metadata results and summary into structured files.
    Returns the paths to the Excel and JSON files.
    """
    excel_path = OUTPUT_DIR / f"metadata_output_{timestamp}.xlsx"
    json_path = OUTPUT_DIR / f"metadata_output_{timestamp}.json"

    logs = []

    try:
        df = pd.DataFrame(results_obj["results"])
        df_flat = pd.json_normalize(results_obj["results"], sep='_')
        df_flat.to_excel(excel_path, index=False, engine='openpyxl')
        logs.append(f"✅ Excel results saved to: {excel_path}")
    except Exception as excel_e:
        logs.append(f"❌ Error saving Excel file: {excel_e}")
        excel_path = None

    try:
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(results_obj, f, ensure_ascii=False, indent=2)
        logs.append(f"✅ JSON results saved to: {json_path}")
    except Exception as json_e:
        logs.append(f"❌ Error saving JSON file: {json_e}")
        json_path = None

    return str(json_path), str(excel_path)

print("Chunking function defined.")

Chunking function defined.


In [ ]:
#@title # Cell 7: OpenAI Interaction (with Content Caching)

# --- REDUCE SYSTEM PROMPT ---
REDUCE_SYSTEM_PROMPT = """
You will receive a JSON array containing multiple JSON objects.
Each object represents partial metadata extracted from a document chunk.
Your task is to synthesize these results into a single, coherent JSON object that represents the complete document.
Instructions:
1. For list fields, merge unique items.
2. For single-value fields, choose the most complete or consistent value.
3. Synthesize text fields (like summary) into a unified version.
Respond ONLY with the final JSON object—no extra text.
"""

# --- INVOKE OPENAI MODEL FUNCTION ---
def invoke_openai_model(system_prompt_user_schema, user_content, model, temperature, log_list, client):
    """
    Invokes the OpenAI Chat Completion API and handles response parsing.

    Parameters:
      - system_prompt_user_schema (str): Extraction prompt or reduce prompt.
      - user_content (dict/list/str): The input content.
      - model (str): Model identifier (e.g., gpt-4o-mini, o1-mini, o3-mini).
      - temperature (float): Temperature setting.
      - log_list (list): List to which log messages are appended.
      - client: Initialized OpenAI client.

    Returns:
      - dict: Parsed JSON output from the API call, or None if parsing fails.
    """
    metadata_text = ""
    try:
        # Check if this is a reduce call (the prompt indicates an array of JSON objects)
        is_reduce_call = "You will receive a JSON array" in system_prompt_user_schema
        if is_reduce_call:
            full_system_prompt = system_prompt_user_schema
        else:
            if 'BASE_SYSTEM_PROMPT' not in globals():
                log_list.append("ERROR: BASE_SYSTEM_PROMPT not defined in global scope!")
                return None
            full_system_prompt = f"{BASE_SYSTEM_PROMPT}\n{system_prompt_user_schema.strip()}"
            full_system_prompt += "\n\nIMPORTANT: Respond ONLY in valid JSON format. Do NOT include any extra text."

        # Convert user content to string
        if isinstance(user_content, (dict, list)):
            user_content_str = json.dumps(user_content, ensure_ascii=False, indent=2)
        else:
            user_content_str = str(user_content)

        api_args = {"model": model}

        # Handle model-specific parameters
        if model == "o1-mini":
            log_list.append("Using combined prompt format for o1-mini; forcing temperature=1.0.")
            combined_content = f"{full_system_prompt}\n\n--- DOCUMENT CONTENT START ---\n{user_content_str}\n--- DOCUMENT CONTENT END ---\n\nREMINDER: Your answer must be solely valid JSON."
            api_args["messages"] = [{"role": "user", "content": combined_content}]
            api_args["temperature"] = 1.0
        elif model == "o3-mini":
            log_list.append("Using standard prompt format for o3-mini (omitting temperature, including response_format).")
            api_args["messages"] = [
                {"role": "system", "content": full_system_prompt},
                {"role": "user", "content": user_content_str}
            ]
            api_args["response_format"] = {"type": "json_object"}
        else:
            log_list.append("Using standard prompt format for model; setting provided temperature and including response_format.")
            api_args["messages"] = [
                {"role": "system", "content": full_system_prompt},
                {"role": "user", "content": user_content_str}
            ]
            api_args["temperature"] = float(temperature)
            api_args["response_format"] = {"type": "json_object"}

        # Log API call parameters (excluding the actual message for brevity)
        log_list.append(f"Calling OpenAI model {model} with arguments: " +
                        f"{ {k: v for k, v in api_args.items() if k != 'messages'} }")
        response = client.chat.completions.create(**api_args)
        metadata_text = response.choices[0].message.content

        try:
            metadata_json = json.loads(metadata_text)
            log_list.append("✅ OpenAI response successfully parsed via direct JSON loading.")
            return metadata_json
        except json.JSONDecodeError as json_e:
            log_list.append(f"❌ Direct JSON parsing failed: {json_e}. Attempting fallback extraction.")
            processed_text = metadata_text.strip()
            if processed_text.startswith("```json"):
                processed_text = processed_text[7:]
            if processed_text.endswith("```"):
                processed_text = processed_text[:-3]
            processed_text = processed_text.strip()
            json_start = processed_text.find('{')
            json_end = processed_text.rfind('}') + 1
            if json_start != -1 and json_end != -1 and json_start < json_end:
                corrected_json_text = processed_text[json_start:json_end]
                try:
                    metadata_json = json.loads(corrected_json_text)
                    log_list.append("✅ Fallback JSON extraction succeeded.")
                    return metadata_json
                except json.JSONDecodeError as fallback_json_e:
                    log_list.append(f"❌ Fallback JSON parsing failed: {fallback_json_e}")
                    return None
            else:
                log_list.append("❌ Fallback extraction failed: Valid JSON boundaries not found.")
                return None
    except openai.AuthenticationError:
        log_list.append("❌ OpenAI Authentication Error encountered during API call.")
        raise
    except openai.RateLimitError:
        log_list.append("❌ OpenAI Rate Limit Error encountered. Consider adding delays or checking your quota.")
        return None
    except openai.APIError as api_e:
        error_details = api_e.body if hasattr(api_e, 'body') else str(api_e)
        log_list.append(f"❌ OpenAI API Error: {error_details}")
        return None
    except Exception as e:
        log_list.append(f"❌ Unexpected error during OpenAI call: {e}")
        log_list.append(traceback.format_exc())
        return None

print("OpenAI interaction pipeline defined.")

OpenAI interaction pipeline defined.


In [ ]:
#@title # Cell 8: Main Pipeline Function

# --- MAIN PIPELINE FUNCTION ---
def run_pipeline_auto_chunk(api_key, model, temp, system_prompt, uploaded_files, progress=gr.Progress(track_tqdm=True)):
    """
    Main pipeline function that processes uploaded PDF files.
    It uses caching to avoid re-extraction, estimates token counts to choose between
    a single API call vs. a chunking (Map-Reduce) strategy, calls OpenAI accordingly,
    and then saves the results (JSON and Excel) in the output folder.

    Parameters:
      - api_key (str): OpenAI API key.
      - model (str): OpenAI model identifier.
      - temp (float): Temperature setting.
      - system_prompt (str): User-defined prompt outlining desired JSON schema.
      - uploaded_files (list): List of Gradio File objects.
      - progress: Gradio progress tracker.

    Returns:
      - Tuple: (excel_path, json_path, log_string)
    """
    import datetime  # For timestamp formatting
    start_time_pipeline = time.time()
    logs = []

    # Record start timestamp with emoji
    start_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    logs.append(f"🚀 Pipeline started at: {start_timestamp}")
    logs.append(f"Cache directory: {str(Path(CACHE_DIR).resolve())}")

    # Initial checks
    if not api_key:
        logs.append("❌ Error: Missing OpenAI API key.")
        return None, None, "\n".join(logs)
    if not system_prompt or not system_prompt.strip():
        logs.append("❌ Error: System prompt is missing or empty.")
        return None, None, "\n".join(logs)
    if not uploaded_files:
        logs.append("❌ Error: No files uploaded.")
        return None, None, "\n".join(logs)
    if model not in PRICING_DATA["prices"]:
        logs.append(f"⚠️ Warning: Model '{model}' not found in pricing data. Proceeding with placeholder pricing.")
        PRICING_DATA["prices"][model] = {"input": 0.0, "output": 0.0}

    # Initialize OpenAI client
    try:
        client = openai.OpenAI(api_key=api_key)
        client.models.list()  # Test authentication
        logs.append("✅ OpenAI client initialized and authenticated.")
    except openai.AuthenticationError:
        logs.append("❌ OpenAI Authentication Error: Check your API key or credits.")
        return None, None, "\n".join(logs)
    except Exception as e:
        logs.append(f"❌ Error initializing OpenAI client: {e}")
        return None, None, "\n".join(logs)

    final_results_list = []
    total_input_tokens_pipeline = 0
    total_output_tokens_pipeline = 0
    files_processed_count = 0
    files_cached_extraction_count = 0
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # For output files, these variables are built here but the helper will write in the correct folder.
    excel_path = os.path.join(OUTPUT_DIR, f"metadata_output_{timestamp}.xlsx")
    json_path = os.path.join(OUTPUT_DIR, f"metadata_output_{timestamp}.json")

    # Process each uploaded file
    for file_obj in progress.tqdm(uploaded_files, desc="Processing PDFs"):
        file_path = file_obj.name
        file_basename = os.path.basename(file_path)
        logs.append(f"\n===== Processing file: {file_basename} =====")

        # --- Check Cache ---
        file_hash = get_file_hash(file_path)
        cache_filepath = None
        extracted_data = None
        extraction_skipped = False

        if file_hash:
            cache_filename = f"{file_hash}_content.json"
            cache_filepath = os.path.join(str(CACHE_DIR), cache_filename)
            if os.path.exists(cache_filepath):
                logs.append(f"Cache hit for {file_basename} (Hash: {file_hash[:8]}...). Loading cached content.")
                try:
                    with open(cache_filepath, 'r', encoding='utf-8') as f:
                        extracted_data = json.load(f)
                    if isinstance(extracted_data, dict) and 'document_markdown' in extracted_data:
                        extraction_skipped = True
                        files_cached_extraction_count += 1
                        logs.append(f"✅ Cached content loaded for {file_basename}.")
                    else:
                        logs.append("❌ Cached file is invalid. Re-extracting content.")
                        os.remove(cache_filepath)
                        extracted_data = None
                except Exception as ex:
                    logs.append(f"❌ Error reading cache for {file_basename}: {ex}. Re-extracting content.")
                    extracted_data = None
            else:
                logs.append(f"❌ No cache found for {file_basename}. Extraction required.")
        else:
            logs.append(f"❌ Could not generate hash for {file_basename}; skipping cache usage.")

        files_processed_count += 1
        file_input_tokens = 0
        file_output_tokens = 0
        file_start_time = time.time()
        file_result = None

        try:
            # --- Content Extraction ---
            if not extraction_skipped:
                logs.append(f"Extracting content for {file_basename}...")
                extraction_logs = []
                extracted_data = extract_structured_content(file_path, extraction_logs)
                logs.extend(extraction_logs)
                if extracted_data and isinstance(extracted_data, dict):
                    logs.append(f"✅ Content extraction successful for {file_basename}.")
                    if cache_filepath:
                        try:
                            os.makedirs(CACHE_DIR, exist_ok=True)
                            with open(cache_filepath, 'w', encoding='utf-8') as f:
                                json.dump(extracted_data, f, ensure_ascii=False, indent=2)
                            logs.append(f"📁 Extracted content saved to cache for {file_basename}.")
                        except Exception as ce:
                            logs.append(f"❌ Error saving cache for {file_basename}: {ce}")
                else:
                    logs.append(f"❌ Extraction failed for {file_basename}. Skipping AI call.")
                    file_result = {"❌ source_file": file_basename, "error": "Content extraction failed"}
                    final_results_list.append(file_result)
                    continue

            if not extracted_data or not isinstance(extracted_data, dict):
                logs.append(f"❌ Critical error: Missing valid extracted data for {file_basename}. Skipping.")
                file_result = {"❌ source_file": file_basename, "error": "Missing extracted data"}
                final_results_list.append(file_result)
                continue

            # --- Token Estimation ---
            content_str = json.dumps(extracted_data, ensure_ascii=False)
            estimated_content_tokens = count_tokens(content_str, model)
            combined_prompt = f"{BASE_SYSTEM_PROMPT}\n{system_prompt.strip()}"
            estimated_prompt_tokens = count_tokens(combined_prompt, model)
            estimated_total_tokens = estimated_content_tokens + estimated_prompt_tokens
            logs.append(f"Estimated tokens for {file_basename}: Total = {estimated_total_tokens} (Prompt = {estimated_prompt_tokens}, Content = {estimated_content_tokens}).")

            # --- Strategy Selection: Single API Call or Chunking ---
            if estimated_total_tokens < CHUNK_THRESHOLD:
                logs.append("Using Single API Call strategy.")
                metadata = invoke_openai_model(system_prompt, extracted_data, model, temp, logs, client)
                if metadata and isinstance(metadata, dict):
                    response_str = json.dumps(metadata, ensure_ascii=False)
                    output_tokens = count_tokens(response_str, model)
                    file_input_tokens = estimated_total_tokens
                    file_output_tokens = output_tokens
                    metadata["source_file"] = file_basename
                    file_result = metadata
                    logs.append(f"✅ Single API Call successful for {file_basename}.")
                else:
                    logs.append(f"❌ Single API call failed for {file_basename}.")
                    file_result = {"❌ source_file": file_basename, "error": "Single call OpenAI/JSON error"}
            else:
                logs.append(f"Using Chunking strategy for {file_basename} (Total tokens {estimated_total_tokens} exceed threshold {CHUNK_THRESHOLD}).")
                partial_results = []
                map_input_total = 0
                map_output_total = 0

                page_logs = []
                pages_data = extract_content_per_page(file_path, page_logs)
                logs.extend(page_logs)
                if not pages_data:
                    logs.append(f"❌ Per-page extraction failed for {file_basename}.")
                    file_result = {"❌ source_file": file_basename, "error": "Per-page extraction failed"}
                    final_results_list.append(file_result)
                    continue

                chunk_logs = []
                chunks = create_document_chunks(pages_data, combined_prompt, model, MAX_CHUNK_TOKENS, chunk_logs)
                logs.extend(chunk_logs)
                if not chunks:
                    logs.append(f"❌ Chunk creation failed for {file_basename}.")
                    file_result = {"❌ source_file": file_basename, "error": "Chunk creation failed"}
                    final_results_list.append(file_result)
                    continue

                # MAP PHASE
                logs.append(f"Starting Map Phase with {len(chunks)} chunk(s).")
                for i, chunk in enumerate(chunks):
                    map_logs = []
                    chunk_content = chunk.get("content_dict", {})
                    chunk_str = json.dumps(chunk_content, ensure_ascii=False)
                    chunk_doc_tokens = count_tokens(chunk_str, model)
                    chunk_total_tokens = estimated_prompt_tokens + chunk_doc_tokens
                    partial_metadata = invoke_openai_model(system_prompt, chunk_content, model, temp, map_logs, client)
                    logs.extend(map_logs)
                    if partial_metadata and isinstance(partial_metadata, dict):
                        partial_response = json.dumps(partial_metadata, ensure_ascii=False)
                        chunk_output_tokens = count_tokens(partial_response, model)
                        map_input_total += chunk_total_tokens
                        map_output_total += chunk_output_tokens
                        partial_results.append(partial_metadata)
                        logs.append(f"✅ Chunk {i+1} processed successfully.")
                    else:
                        logs.append(f"❌Chunk {i+1} failed; partial result omitted.")

                if not partial_results:
                    logs.append(f"No valid partial metadata for {file_basename} from Map phase.")
                    file_result = {"source_file": file_basename, "error": "No valid Map phase results"}
                else:
                    logs.append(f"Starting Reduce Phase with {len(partial_results)} partial result(s).")
                    reduce_input_str = json.dumps(partial_results, ensure_ascii=False)
                    reduce_content_tokens = count_tokens(reduce_input_str, DEFAULT_MODEL)
                    reduce_prompt_tokens = count_tokens(REDUCE_SYSTEM_PROMPT, DEFAULT_MODEL)
                    reduce_total_tokens = reduce_content_tokens + reduce_prompt_tokens
                    logs.append(f"Estimated tokens for Reduce phase: {reduce_total_tokens}.")
                    if reduce_total_tokens > REDUCE_TOKEN_LIMIT:
                        logs.append(f"Reduce phase input tokens {reduce_total_tokens} exceed limit {REDUCE_TOKEN_LIMIT}.")
                        file_result = {"source_file": file_basename, "error": "Reduce phase input too large", "partial_results_count": len(partial_results)}
                    else:
                        reduce_logs = []
                        final_metadata = invoke_openai_model(REDUCE_SYSTEM_PROMPT, partial_results, DEFAULT_MODEL, 0.0, reduce_logs, client)
                        logs.extend(reduce_logs)
                        if final_metadata and isinstance(final_metadata, dict):
                            final_response = json.dumps(final_metadata, ensure_ascii=False)
                            reduce_output_tokens = count_tokens(final_response, DEFAULT_MODEL)
                            file_input_tokens += reduce_total_tokens
                            file_output_tokens += reduce_output_tokens
                            final_metadata["source_file"] = file_basename
                            file_result = final_metadata
                            logs.append("✅ Reduce Phase successful.")
                        else:
                            logs.append("❌ Reduce Phase failed.")
                            file_result = {"source_file": file_basename, "error": "Reduce phase failed"}

            if file_result:
                final_results_list.append(file_result)
            else:
                logs.append(f"⚠️ No definitive result for {file_basename}.")
                final_results_list.append({"source_file": file_basename, "error": "No definitive result generated."})

        except openai.AuthenticationError:
            logs.append(f"❌ OpenAI Authentication Error during processing of {file_basename}. Stopping pipeline.")
            final_results_list.append({"❌ source_file": file_basename, "error": "Authentication error"})
            break
        except Exception as e:
            logs.append(f"❌ Unhandled error processing {file_basename}: {e}")
            logs.append(traceback.format_exc())
            final_results_list.append({"source_file": file_basename, "error": f"Unhandled error: {str(e)[:150]}"})

        total_input_tokens_pipeline += file_input_tokens
        total_output_tokens_pipeline += file_output_tokens
        file_time = time.time() - file_start_time
        logs.append(f"Time for {file_basename}: {file_time:.2f} seconds.")
        logs.append(f"Token count for {file_basename}: Input ~{file_input_tokens}, Output ~{file_output_tokens}")

    # --- Final Summary and Save Outputs ---
    pipeline_time = time.time() - start_time_pipeline
    mins, secs = divmod(pipeline_time, 60)
    time_str = f"{int(mins)}m {int(secs)}s" if mins > 0 else f"{int(secs)}s"
    logs.append("\n" + "=" * 50)
    logs.append("FINAL SUMMARY:")
    logs.append(f"Total pipeline time: {time_str}")
    logs.append(f"Files submitted: {len(uploaded_files)}")
    logs.append(f"Files with cached extraction: {files_cached_extraction_count}")
    logs.append(f"Files processed (sent to AI): {files_processed_count}")
    successful = sum(1 for res in final_results_list if isinstance(res, dict) and 'error' not in res)
    failed = len(final_results_list) - successful
    logs.append(f"✅ Successful AI outputs: {successful}")
    logs.append(f"❌ Files with errors: {failed}")
    logs.append(f"Model used: {model}")
    logs.append(f"Total tokens for all AI calls: {total_input_tokens_pipeline + total_output_tokens_pipeline} (Input: {total_input_tokens_pipeline}, Output: {total_output_tokens_pipeline})")
    model_pricing = PRICING_DATA["prices"].get(model, {"input": 0.0, "output": 0.0})
    if model_pricing["input"] == 0.0 and model_pricing["output"] == 0.0:
        logs.append(f"Estimated cost: Not available for model {model}.")
    else:
        cost_input = (total_input_tokens_pipeline / 1_000) * model_pricing["input"]
        cost_output = (total_output_tokens_pipeline / 1_000) * model_pricing["output"]
        logs.append(f"Estimated cost: ${cost_input + cost_output:.4f} (Input: ${cost_input:.4f}, Output: ${cost_output:.4f})")
    logs.append("=" * 50)

    if not final_results_list:
        logs.append("⚠️ No results generated for any file.")
        return None, None, "\n".join(logs)

    valid_results = [res for res in final_results_list if isinstance(res, dict)]
    valid_for_export = [res for res in valid_results if 'error' not in res]

    if not valid_for_export:
        logs.append("⚠️ No valid metadata to save after filtering errors.")
        excel_out, json_out = None, None
    else:
        # Save outputs using the helper function; files will be written in /content/metadata_output/
        json_out, excel_out = save_outputs_to_disk({"results": valid_for_export}, timestamp)
        logs.append(f"📁 Results saved to disk: Excel: {excel_out}, JSON: {json_out}")

    end_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    logs.append(f"✅ Pipeline completed at: {end_timestamp}")

    return excel_out, json_out, "\n".join(logs)

print("OpenAI main pipeline functions defined.")

OpenAI main pipeline functions defined.


In [ ]:
#@title Cell 9: Gradio UI App Launch

# Set default prompt for UI based on external definition
SYSTEM_PROMPT_UI_DEFAULT = SYSTEM_PROMPT

# Function to clear all data from the filesystem (cache and output folders)
def clear_all_data():
    """
    Clears UI components and deletes all files in the cache folder (CACHE_DIR)
    and the metadata output folder (OUTPUT_DIR).

    Returns a tuple to clear the Gradio outputs: (None, None, None, "")
    """
    # Clear the cache folder:
    try:
        for file in Path(CACHE_DIR).iterdir():
            try:
                file.unlink()
            except Exception as e:
                print(f"❌ Error deleting cache file {file}: {e}")
        print("Cache folder cleared.")
    except Exception as e:
        print("❌ Error clearing cache folder:", e)

    # Clear metadata output files in OUTPUT_DIR:
    try:
        output_dir = Path(OUTPUT_DIR)
        if output_dir.exists():
            for file in output_dir.iterdir():
                try:
                    file.unlink()
                except Exception as e:
                    print(f"❌ Error deleting output file {file}: {e}")
            print("✅ Metadata output files cleared.")
    except Exception as e:
        print("❌ Error clearing metadata output files:", e)

    # Return cleared UI outputs (reset uploads, output files, and logs)
    return None, None, None, ""

# Function to reset the UI fields without deleting saved data from disk.
def reset_fields():
    """
    Resets the UI fields to their default values:
      - System prompt set to the default SYSTEM_PROMPT_UI_DEFAULT.
      - Model dropdown set to DEFAULT_MODEL.
      - Temperature reset to default (0.0).
      - Clears the file upload and results and resets the logs.

    Returns a tuple matching the outputs for (uploaded, excel_output, json_output, log_output).
    """
    return None, None, None, "", SYSTEM_PROMPT_UI_DEFAULT, DEFAULT_MODEL, 0.0

with gr.Blocks(title="📚 AI PDF Metadata Extractor") as demo:
    gr.Markdown("## 📚 AI PDF Metadata Extractor by AI Academy")
    gr.Markdown("Upload your PDF files below to extract metadata (including text, tables, and OCR content).")

    with gr.Row(equal_height=True):
        # Left column: Inputs
        with gr.Column(scale=1):
            system_prompt_input = gr.Textbox(
                label="📝 System Prompt (Define JSON structure)",
                lines=18,
                value=SYSTEM_PROMPT_UI_DEFAULT,
                placeholder="Enter your system prompt (defining the JSON schema) here."
            )
            api_key = gr.Textbox(
                label="🔐 OpenAI API Key",
                type="password",
                placeholder="Enter your OpenAI API key (e.g., sk-...)"
            )
            model = gr.Dropdown(
                choices=model_choices,
                value=DEFAULT_MODEL,
                label="🧠 OpenAI Model"
            )
            temp = gr.Slider(
                minimum=0.0,
                maximum=1.0,
                value=0.0,
                step=0.1,
                label="🎛️ Temperature (0 = deterministic, 1 = creative)"
            )
        # Right column: Uploads, Controls, Logs, and Results
        with gr.Column(scale=1):
            uploaded = gr.File(
                label="📄 Upload PDF(s)",
                file_types=[".pdf"],
                file_count="multiple",
                height=150
            )
            with gr.Row():
                submit_btn = gr.Button("🚀 Extract Metadata", variant="primary")
                reset_btn = gr.Button("🔄 Reset Fields")
                clear_btn = gr.Button("🧹 Clear Data")
            log_output = gr.Textbox(
                label="🪵 Processing Logs",
                lines=8,
                interactive=False,
                show_copy_button=True
            )
            with gr.Group():
                gr.Markdown("### 📥 Download Results")
                with gr.Row():
                    excel_output = gr.File(
                        label="Excel (.xlsx)",
                        file_types=[".xlsx"],
                        interactive=False
                    )
                    json_output = gr.File(
                        label="JSON (.json)",
                        file_types=[".json"],
                        interactive=False
                    )

    # --- Event Handlers ---
    submit_btn.click(
        fn=run_pipeline_auto_chunk,
        inputs=[api_key, model, temp, system_prompt_input, uploaded],
        outputs=[excel_output, json_output, log_output]
    )

    # The Reset Fields button only resets UI input fields
    # It returns:
    #   - None for the uploaded files widget,
    #   - None for excel_output,
    #   - None for json_output,
    #   - an empty log,
    #   - the default system prompt,
    #   - the default model,
    #   - and a default temperature value.
    reset_btn.click(
        fn=reset_fields,
        inputs=[],   # No inputs needed.
        outputs=[uploaded, excel_output, json_output, log_output, system_prompt_input, model, temp]
    )

    # The Clear Data button deletes files from disk (cache and output folders) and then clears UI fields.
    clear_btn.click(
        fn=clear_all_data,
        inputs=[],  # No inputs needed for clearing
        outputs=[uploaded, excel_output, json_output, log_output]
    )

# --- Launch the App ---
if __name__ == "__main__":
    try:
        _ = SYSTEM_PROMPT  ##############
        print("✅ SYSTEM_PROMPT is defined. Launching Gradio App...")
    except NameError:
        print("⚠️ WARNING: SYSTEM_PROMPT is not defined. Please define it or input it directly into the UI before running.")
    demo.launch(debug=True, share=True)


✅ SYSTEM_PROMPT is defined. Launching Gradio App...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://69490e1d007a084cac.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://69490e1d007a084cac.gradio.live
